# TCAA Experiment — Google Colab（三目标闭环 / 结果全量留存）

**TCAA** = Token-Consumption Amplification Attack：在联邦微调（FFT）里，一个恶意客户端上传精心构造的 LoRA 更新，使聚合后的因果 LLM 在**带触发词的输入**上生成更多 token（更耗算力/成本），同时保持输出正确、并尽量躲过服务器检测。

本 notebook 一次跑通并可视化 TCAA 的**三大目标**，覆盖三条实验路径：
- **① 资源放大**：触发输入 `D_τ` 的成本放大比（均值/中位）与触发选择性；
- **② 性能保持**：干净输入困惑度基本不变，且 **ROUGE-L 召回≈1**（答案内容仍在，加长不是废话）；
- **③ 隐蔽性**：恶意更新落入良性参数空间**距离/余弦**包络（ALM 约束停在边界）。

实验路径：**Step 5–6.5** 单轮 Phase-0（三目标一次测全 + 8 张图）；**Step 7** 多轮 FL durability（放大是否随轮次累积、逐轮是否隐蔽）；**Step 8** Pareto（放大 vs 隐蔽预算的权衡前沿）。

> **⚠️ 结果留存（重要）**：Colab GPU 释放后临时磁盘上的 `results/` 会丢失，**只有已渲染进单元输出的表格与内联图会随 notebook 保存而留在页面**。因此每段实验都会把结果**打印 + 内联出图**；跑完请先 **Ctrl/⌘+S 保存**，再让 Step 10 释放 GPU。

## 使用步骤
1. **开启 GPU**：Runtime → Change runtime type → **GPU**（Qwen2.5-0.5B 用 T4 即可；1.5B/3B 建议 L4/A100）
2. **Step 0**：让 Colab 拿到含 `tcaa/` 的代码（三选一，见下）
3. Runtime → **Run all**（多轮 FL / Pareto 较重，可按各自单元的注释调小规模）
4. 看 **Step 6** 总览卡、**Step 6.5/7/8** 内联图，**Ctrl/⌘+S 保存**，**Step 9** 下载 `results/`

> 本 notebook 跑 `tcaa/` 包（TCAA）；AugMP 分类基线已移到 `augmp_baseline/` 子目录，用 `cd augmp_baseline && python main.py` 运行——它与 TCAA 互不影响（`num_attackers` 等 config 各管各的）。

## Step 0: Fetch Code（获取代码 · 三选一）

含 `tcaa/` 的代码目前只在你本地，需先让 Colab 能访问。**任选一种**：

- **A｜GitHub（推荐）**：把本仓库 push 到你自己的 GitHub，然后把下面 `REPO_URL` 改成你的地址。
- **B｜Google Drive**：把整个 `TCA-Attacker` 文件夹放到 Drive，运行下面「可选：挂载 Drive」单元并改成你的路径。
- **C｜手动上传**：把整个文件夹拖到 Colab 左侧文件区。

下面的取码单元会**自动按 A/B/C 顺序探测**，找到 `tcaa/` 即用。

In [ ]:
# （可选 · 方式 B）把代码放在 Google Drive 时才用：取消下面两行注释并改成你的实际路径，再运行本单元。
# from google.colab import drive; drive.mount('/content/drive')
# %cd '/content/drive/MyDrive/TCA-Attacker'
print('如走 Google Drive：请取消上面两行注释并改路径；否则跳过本单元。')

In [ ]:
# 自动获取代码：C(当前目录已有) -> A(GitHub 克隆)。找到 tcaa/ 即停。
import os, sys, subprocess
from pathlib import Path

# 👉 方式 A：改成你自己的 fork（必须包含 tcaa/ 包）。默认按 git user 猜测，请自行确认。
REPO_URL = 'https://github.com/GuangLun2000/TCA-Attacker.git'

def find_repo_root():
    for base in [Path('.'), *sorted(Path('.').glob('*')), *sorted(Path('.').glob('*/*'))]:
        if base.is_dir() and (base / 'tcaa' / 'phase0_runner.py').exists():
            return base
    return None

root = find_repo_root()
if root is None and '<' not in REPO_URL:
    print(f'📥 未在当前环境找到 tcaa/，尝试克隆 {REPO_URL} ...')
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)
        root = find_repo_root()
    except subprocess.CalledProcessError:
        print('❌ 克隆失败（仓库私有/地址错/无权限）。')

if root is None:
    raise SystemExit(
        '❌ 未找到 tcaa/ 包。请用以下任一方式让 Colab 拿到代码：\n'
        '  A) 把仓库 push 到你的 GitHub，并把 REPO_URL 改对；\n'
        '  B) 把文件夹放 Google Drive，运行上一单元挂载并 cd 过去；\n'
        '  C) 把整个文件夹上传到 Colab 左侧文件区，然后重跑本单元。')

os.chdir(root.resolve())
sys.path.insert(0, str(Path('.').resolve()))
print('✅ 使用仓库:', Path('.').resolve())
assert Path('tcaa/phase0_runner.py').exists()

## Step 1: Install Dependencies（安装依赖）

In [ ]:
# Colab 已自带 torch；补装 peft + datasets + 较新的 transformers。
%pip install -q "transformers>=4.37,<5" "peft>=0.6" "datasets>=2" tqdm
# Colab 预装的旧版 torchao(0.10) 会让新版 peft 的 LoRA 分发器报 ImportError；本项目不用 torchao，卸载即可。
!pip uninstall -y torchao 2>/dev/null || true
print('✅ 依赖安装完成（已移除不兼容的 torchao）。')

## Step 2: Verify Files and GPU（检查文件与 GPU）

In [ ]:
import torch
from pathlib import Path

need = ['tcaa/phase0_runner.py', 'tcaa/length_surrogate.py', 'tcaa/cost_model.py',
        'tcaa/causal_model.py', 'tcaa/gen_data.py', 'tcaa/stealth.py', 'tcaa/metrics.py',
        'tcaa/alm.py', 'tcaa/visualize.py', 'tcaa/fl_runner.py', 'tcaa/pareto_runner.py']
missing = [f for f in need if not Path(f).exists()]
print('⚠️ 缺少文件:', missing) if missing else print('✅ tcaa/ 关键文件齐全。')

print('\nPyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} ({p.total_memory/1e9:.1f} GB)')
else:
    print('⚠️ 未检测到 GPU。Runtime → Change runtime type → GPU（不然会很慢）')

## Step 3: Sanity Check（单元测试，快速自检环境）

In [ ]:
!python -m tcaa.tests.test_length_surrogate
!python -m tcaa.tests.test_alm
!python -m tcaa.tests.test_stealth_matches_server

## Step 4: Configure（正式实验配置）

默认：**Qwen2.5-0.5B（base）+ Alpaca 指令数据集**，正式规模。Alpaca 是指令微调的标准基准、输出长度自由，最适合展示 token 膨胀（也满足 Spec §8「开放式生成」的要求）。换骨干/数据集见注释；改完直接往下 Run。

In [ ]:
TCAA_CONFIG = {
    "experiment_name": "tcaa_qwen25_alpaca",
    "backbone": "Qwen/Qwen2.5-0.5B",         # base 版做联邦微调；仅 decoder-only（会校验）
    "source": "alpaca",                       # 'alpaca'(指令,默认) | 'dolly' | 'xsum' | 'cnn_dailymail'
    "reference_source": "dataset",            # 或 'benign_verbose'（Spec §3 source ii）

    # —— 联邦设置（与 AugMP 约定一致，便于对比）——
    "num_clients": 5, "num_attackers": 1, "local_epochs": 2,
    "dirichlet_alpha": 0.3, "server_lr": 1.0,
    "client_lr": 1e-4, "batch_size": 8, "grad_clip_norm": 1.0,
    "warmup_steps": 0,                        # 预训练骨干无需 warm-up

    # —— TCAA 攻击（含逻辑改进）——
    "gamma": 1.0, "attacker_lr": 1e-4, "attacker_steps": 300,
    "gamma_clean": 0.5,                       # clean 长度锚：抑制泄漏、提升触发选择性（单边 hinge 到基线）
    "use_onpolicy_length": True,              # 沿模型自身 greedy 续写优化 E[L]（对齐"目标=推理测量"，修 exposure bias）
    "onpolicy_horizon": 96,                   # on-policy 续写步数；每步一次 generate，越大越慢
    "use_fallback_surrogate": False,

    # —— 成本模型 / 生成 ——
    # c_f/c_a 只有比值影响超线性阈值；如需硬件标定用 cost_model.calibrate_coefficients(d_model)
    "c_f": 1.0, "c_a": 1.0, "max_new_tokens": 256,   # 上限抬到 256：暴露被截断的真实放大 + 争取进入超线性区
    "num_dump_examples": 4,                   # 定性样例数/split（触发→长文、clean→正常）

    # —— 隐蔽阈值：None = 用良性包络（d_T=良性最大距离，δ_T=良性最小余弦）——
    "d_T": None, "delta_T": None,

    # —— 数据规模（正式）——
    "pool_size": 512, "eval_size": 64,
    "lora_r": 8, "lora_alpha": 16,
}

# ===== 常用替换（取消注释即可）=====
# 更大骨干（需 L4/A100）：
# TCAA_CONFIG.update({"backbone": "Qwen/Qwen2.5-1.5B"})
# TCAA_CONFIG.update({"backbone": "Qwen/Qwen2.5-3B"})
# 换数据集：
# TCAA_CONFIG.update({"source": "dolly"})       # 另一指令数据集
# TCAA_CONFIG.update({"source": "xsum"})        # 摘要任务
# 详细但正确的参考（source ii，进一步缓解短参考问题）：
# TCAA_CONFIG.update({"reference_source": "benign_verbose"})
# on-policy 太慢时改回 teacher-forced（更快、但目标与推理略脱节）：
# TCAA_CONFIG.update({"use_onpolicy_length": False})
# 更快试跑：TCAA_CONFIG.update({"pool_size":128,"eval_size":32,"attacker_steps":150,"local_epochs":1,"onpolicy_horizon":64})
# 更强攻击：提高 gamma（如 2~4）与 attacker_steps；注意监控效用/重复率与 train-vs-inference 长度差

print('配置就绪:', TCAA_CONFIG['backbone'], '+', TCAA_CONFIG['source'],
      '| pool', TCAA_CONFIG['pool_size'], '| steps', TCAA_CONFIG['attacker_steps'],
      '| on-policy', TCAA_CONFIG['use_onpolicy_length'], '| max_new', TCAA_CONFIG['max_new_tokens'])

## Step 5: Run（运行正式实验）

In [ ]:
import time, warnings
warnings.filterwarnings('ignore')
from tcaa.phase0_runner import run_phase0

print('🚀 TCAA Stage A 开始 ...'); print('=' * 60)
t0 = time.time()
results = run_phase0(TCAA_CONFIG)
print(f'\n✅ 完成，用时 {(time.time()-t0)/60:.1f} 分钟。结果写入 results/tcaa_phase0/')

## Step 6: View Results（结果总览 + 三目标判定 · 内联留存）

打印结果表与关键数字（含 **ROUGE-L 召回**这条效用证据）、定性样例，并渲染一张**三目标 HTML 总览卡**。这些都是纯文本/静态 HTML 输出，会随 notebook 保存而留在页面。

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
from tcaa.visualize import summary_html

md_path = Path('results/tcaa_phase0/phase0_results.md')
if md_path.exists():
    print(md_path.read_text())

c, u, s = results['cost'], results['utility'], results['stealth']
bt, at = c['baseline_tau'], c['attacked_tau']
print('\n—— 关键数字 ——')
print(f"① 放大比 τ 均值={c['amplification_tau']}x  中位(抗截断)={c.get('amplification_tau_median')}x  "
      f"clean={c['amplification_clean']}x  选择性={c['trigger_selectivity']}x")
print(f"   输出长度 τ: {bt['mean_output_len']} -> {at['mean_output_len']}  "
      f"| clean: {c['baseline_clean']['mean_output_len']} -> {c['attacked_clean']['mean_output_len']}")
print(f"   截断率 τ(cap命中): {bt.get('truncation_rate')} -> {at.get('truncation_rate')}  "
      f"| 重复率 τ(退化): {bt.get('mean_repetition')} -> {at.get('mean_repetition')}")
print(f"② 效用 ppl 干净比值={u['ppl_clean_ratio']} (≈1 保持)  "
      f"| ROUGE-L 召回 τ: {u.get('rouge_recall_tau_baseline')} -> {u.get('rouge_recall_tau_attacked')} "
      f"(×{u.get('rouge_recall_tau_ratio')}, ≈1 表示答案内容仍在)")
print(f"③ 隐蔽: 距离 {s['attacker_distance']} vs d_T {s['d_T']} -> "
      f"{'满足' if s['distance_satisfied'] else '违反'}; "
      f"余弦 {s['attacker_cosine']} vs δ_T {s['delta_T']} -> "
      f"{'满足' if s['cosine_satisfied'] else '违反'}; "
      f"联合满足={s['jointly_satisfied']}")

# 定性样例（触发→长文，clean→正常；ROUGE 召回看"加长但答案仍在"）
ex_path = Path('results/tcaa_phase0/examples.jsonl')
if ex_path.exists():
    import json
    print('\n—— 定性样例（前若干条）——')
    for line in ex_path.read_text().splitlines()[:4]:
        e = json.loads(line)
        print(f"[{e['split']}] len {e['baseline_len']} -> {e['attacked_len']}  "
              f"| ROUGE召回 {e.get('baseline_rouge_recall')} -> {e.get('attacked_rouge_recall')}")
        print(f"    prompt : {e['prompt'][:90]}")
        print(f"    attacked: {e['attacked_output'][:140]}")

# 三目标 HTML 总览卡（静态 HTML，释放 GPU 后仍留在页面）
display(HTML(summary_html(results)))

## Step 6.5: Visualize（可视化 · 内联显示到页面）

内联渲染 **8 张出版级图**：三目标总览、成本放大、输出长度分布、效用（困惑度 + ROUGE 召回）、参数空间隐蔽性、攻击优化轨迹、**ALM 隐蔽约束收敛**、成本模型。图以 base64 嵌入单元输出，随 notebook 保存而留存（同时也存了一份到 `results/tcaa_phase0/figures/`）。

In [ ]:
%matplotlib inline
from tcaa.visualize import render_report

# 内联渲染全部图表；results 来自 Step 5（从内存字典取数，不依赖磁盘）
_figs = render_report(results)
print(f"\n✅ 已内联显示 {len(_figs)} 张图（并保存到 results/tcaa_phase0/figures/）")

## Step 7: 多轮联邦 durability（可选 · 较重）

多轮 FL 才能回答单轮看不到的两件事：放大是否**随通信轮次累积/存活**（durability），以及客户端采样下**每一轮**是否都留在良性隐蔽包络内。下面并行跑一条「良性-only」全局作为成本基线。

> **较重**：默认已调小（20 轮、pool 1500）便于一次跑完；要复现正式规模（5 benign + 2 attackers、50 轮、pool 4000）取消注释即可。运行中的 durability/stealth 表会打印到本单元输出并随页面留存。

In [ ]:
import time
from tcaa.fl_runner import run_fl, default_fl_config
from tcaa.visualize import render_fl_report

FL_CONFIG = default_fl_config()
FL_CONFIG.update({
    "backbone": TCAA_CONFIG["backbone"], "source": TCAA_CONFIG["source"],
    "num_clients": 7, "num_attackers": 2,   # 5 benign + 2 coordinated attackers
    # —— 默认调小以便一次跑完（可上调）——
    "num_rounds": 20, "clients_per_round": 5, "measure_every": 5,
    "pool_size": 1500, "eval_size": 128, "attacker_steps": 60,
})
# 正式规模（论文级，较慢）：
# FL_CONFIG.update({"num_rounds": 50, "pool_size": 4000, "eval_size": 256, "attacker_steps": 100})

print('🚀 多轮 FL 开始 ...'); print('=' * 60)
t0 = time.time()
fl_results = run_fl(FL_CONFIG)      # 其 durability/stealth 表已打印到输出（留存）
print(f'\n✅ 多轮完成，用时 {(time.time()-t0)/60:.1f} 分钟。')

%matplotlib inline
_ = render_fl_report(fl_results)    # durability（堆叠子图）+ 逐轮隐蔽图，内联留存

## Step 8: Pareto —— 放大 vs 隐蔽预算权衡（可选 · 较重）

扫 `gamma × gamma_clean × kappa`（kappa = 允许占用的良性距离包络比例），画出**放大-隐蔽前沿**：预算收紧时放大是否塌陷、有没有既隐蔽又放大的可行点。每个网格点是一次约束版 Phase-0。

> **较重**：默认小网格（2 个 γ × 3 个 κ = 6 点）并调小单点规模；要更密的前沿改 `gammas/kappas` 与 `PARETO_BASE`。前沿表会打印到输出并随页面留存。

In [ ]:
import time
from tcaa.pareto_runner import run_pareto
from tcaa.visualize import render_pareto_report

PARETO_BASE = dict(TCAA_CONFIG)
PARETO_BASE.update({          # 单点调小，保持网格可跑完
    "pool_size": 256, "eval_size": 48, "attacker_steps": 120,
    "results_subdir": "tcaa_phase0",
})
print('🚀 Pareto 扫描开始 ...'); print('=' * 60)
t0 = time.time()
pareto = run_pareto(PARETO_BASE, gammas=[1.0, 2.0], gamma_cleans=[0.5],
                    kappas=[0.6, 0.8, 1.0])   # 前沿表已打印到输出（留存）
print(f'\n✅ Pareto 完成，用时 {(time.time()-t0)/60:.1f} 分钟。')

%matplotlib inline
_ = render_pareto_report(pareto)   # 前沿 + κ 权衡图，内联留存

## Step 9: Download Results（打包下载 · 离线备份）

页面留存已够用；这里再把 `results/`（phase0 / 多轮 FL / pareto 三段）打包成 zip 供离线备份。

In [ ]:
import zipfile
from pathlib import Path

rd = Path('results')
zp = 'tcaa_results.zip'
if rd.exists():
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in rd.rglob('*'):
            if f.is_file():
                z.write(f, f.relative_to(rd))
    print('✅ 已打包 results/（phase0 + fl + pareto）:', zp)
    try:
        from google.colab import files
        files.download(zp)
    except Exception:
        print('本地打开:', Path(zp).resolve())
else:
    print('⚠️ 未找到 results/，请先运行前面的实验单元。')

## Step 10: 自动释放 GPU（运行结束 30 秒后）

**释放前请先 Ctrl/⌘+S 保存 notebook**，否则内联的表与图不会写回 `.ipynb`。本单元**倒计时 30 秒**再自动释放 Colab GPU（`runtime.unassign()`），避免空占浪费；释放会**断开当前 runtime**，务必让 Step 9 下载先完成。

> 想继续使用 GPU：在这 30 秒倒计时内点击 Colab 工具栏的 **■（停止本单元）** 即可取消释放。

In [ ]:
# 运行结束 → 倒计时 60 秒 → 自动释放 Colab GPU（避免空占浪费）。
# ⚠️ 释放前请先 Ctrl/⌘+S 保存 notebook，确保内联的表与图写回 .ipynb。
# 取消自动释放：在这 60 秒内点击工具栏的 ■ 停止本单元即可。
import time

WAIT_SECONDS = 60
print(f'✅ 实验与下载已完成。请先 Ctrl/⌘+S 保存；将在 {WAIT_SECONDS} 秒后自动释放 GPU。')
print('   如需继续使用 GPU：现在点击工具栏的 ■（停止本单元）即可取消释放。')
try:
    for i in range(WAIT_SECONDS, 0, -1):
        print(f'\r   {i:2d} 秒后释放 GPU ...  ', end='', flush=True)
        time.sleep(1)
    print('\n🔌 正在释放 Colab GPU runtime ...')
    from google.colab import runtime
    runtime.unassign()          # 断开并释放 GPU（会结束当前会话）
except KeyboardInterrupt:
    print('\n⏹️  已取消自动释放，GPU 继续保留。')
except Exception as e:
    print('\n（非 Colab 环境或释放失败，可忽略）:', e)